<a href="https://colab.research.google.com/github/Mehaha-sys/data-ml-portfolio/blob/main/Exoplanet_Candidate_Classification.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
from pathlib import Path
import shutil

# Project location
PROJECT_DIR = Path(
    "/content/drive/MyDrive/Exoplanet-Candidate-Classification"
)

# Original dataset — NEVER MODIFY
RAW_FILE = PROJECT_DIR / "kepler_koi_dr25.csv"

# Folder for our working copy
WORKING_DIR = PROJECT_DIR / "working_data"
WORKING_DIR.mkdir(exist_ok=True)

# Working copy
WORKING_FILE = WORKING_DIR / "kepler_koi_working.csv"

print("Raw file exists:", RAW_FILE.exists())
print("Raw file:", RAW_FILE)

Raw file exists: True
Raw file: /content/drive/MyDrive/Exoplanet-Candidate-Classification/kepler_koi_dr25.csv


In [ ]:
if not WORKING_FILE.exists():
    shutil.copy2(RAW_FILE, WORKING_FILE)
    print("Working copy created.")
else:
    print("Working copy already exists — keeping the existing copy.")

print("Working file:", WORKING_FILE)

Working copy created.
Working file: /content/drive/MyDrive/Exoplanet-Candidate-Classification/working_data/kepler_koi_working.csv


In [ ]:
import pandas as pd

df = pd.read_csv(WORKING_FILE, comment="#")

print("Dataset shape:", df.shape)

Dataset shape: (8054, 141)


In [ ]:
# Build a column inventory

column_info = pd.DataFrame({
    "column": df.columns,
    "dtype": df.dtypes.astype(str).values,
    "missing_count": df.isna().sum().values,
    "missing_%": (df.isna().mean() * 100).round(2).values,
    "unique_values": df.nunique(dropna=True).values
})

display(column_info)

,column,dtype,missing_count,missing_%,unique_values
0,rowid,int64,0,0.00,8054
1,kepid,int64,0,0.00,6923
2,kepoi_name,object,0,0.00,8054
3,kepler_name,object,5323,66.09,2731
4,koi_disposition,object,0,0.00,3
...,...,...,...,...,...
136,koi_dikco_mra_err,float64,166,2.06,160
137,koi_dikco_mdec,float64,166,2.06,2094
138,koi_dikco_mdec_err,float64,166,2.06,161
139,koi_dikco_msky,float64,166,2.06,1784


In [ ]:
# What are the actual disposition categories?

print(df["koi_disposition"].value_counts(dropna=False))

koi_disposition
FALSE POSITIVE    3965
CONFIRMED         2731
CANDIDATE         1358
Name: count, dtype: int64


In [ ]:
# Keep only confirmed and false-positive objects

df_binary = df[
    df["koi_disposition"].isin(["CONFIRMED", "FALSE POSITIVE"])
].copy()

print("Binary dataset shape:", df_binary.shape)

print("\nDisposition counts:")
print(df_binary["koi_disposition"].value_counts())

Binary dataset shape: (6696, 141)

Disposition counts:
koi_disposition
FALSE POSITIVE    3965
CONFIRMED         2731
Name: count, dtype: int64


In [ ]:
candidate_features = [
    # Transit
    "koi_period",
    "koi_eccen",
    "koi_longp",
    "koi_impact",
    "koi_duration",
    "koi_ingress",
    "koi_depth",
    "koi_ror",
    "koi_srho",
    "koi_incl",
    "koi_dor",

    # Signal / detection
    "koi_max_sngle_ev",
    "koi_max_mult_ev",
    "koi_model_snr",
    "koi_num_transits",
    "koi_bin_oedp_sig",

    # Stellar
    "koi_steff",
    "koi_slogg",
    "koi_smet",
    "koi_srad",
    "koi_smass",
    "koi_kepmag",

    # Derived system/planet properties
    "koi_prad",
    "koi_sma",
    "koi_teq",
    "koi_insol"
]

print("Candidate measurements:", len(candidate_features))

for feature in candidate_features:
    print(feature)

Candidate measurements: 26
koi_period
koi_eccen
koi_longp
koi_impact
koi_duration
koi_ingress
koi_depth
koi_ror
koi_srho
koi_incl
koi_dor
koi_max_sngle_ev
koi_max_mult_ev
koi_model_snr
koi_num_transits
koi_bin_oedp_sig
koi_steff
koi_slogg
koi_smet
koi_srad
koi_smass
koi_kepmag
koi_prad
koi_sma
koi_teq
koi_insol


In [ ]:
# Remove measurements that are completely missing
# or contain no usable variation.

valid_features = []

for feature in candidate_features:
    non_missing = df_binary[feature].dropna()

    if len(non_missing) == 0:
        continue

    if non_missing.nunique() <= 1:
        continue

    valid_features.append(feature)

print("Original candidate measurements:", len(candidate_features))
print("Measurements passing basic validity:", len(valid_features))

print("\nRemoved:")
for feature in candidate_features:
    if feature not in valid_features:
        print("-", feature)

Original candidate measurements: 26
Measurements passing basic validity: 23

Removed:
- koi_eccen
- koi_longp
- koi_ingress


In [ ]:
from sklearn.metrics import roc_auc_score

# Encode:
# CONFIRMED = 1
# FALSE POSITIVE = 0

y = (
    df_binary["koi_disposition"] == "CONFIRMED"
).astype(int)

auc_results = []

for feature in valid_features:

    temp = df_binary[[feature]].copy()
    temp["target"] = y

    temp = temp.dropna()

    if temp["target"].nunique() < 2:
        continue

    auc = roc_auc_score(
        temp["target"],
        temp[feature]
    )

    auc_results.append({
        "feature": feature,
        "raw_auc": auc,
        "discrimination_auc": max(auc, 1 - auc),
        "missing_%": df_binary[feature].isna().mean() * 100
    })

auc_df = (
    pd.DataFrame(auc_results)
    .sort_values(
        "discrimination_auc",
        ascending=False
    )
    .reset_index(drop=True)
)

display(auc_df)

,feature,raw_auc,discrimination_auc,missing_%
0,koi_incl,0.740565,0.740565,0.821386
1,koi_smet,0.714167,0.714167,0.896057
2,koi_dor,0.713754,0.713754,0.821386
3,koi_teq,0.300837,0.699163,0.821386
4,koi_insol,0.300846,0.699154,0.821386
5,koi_prad,0.304547,0.695453,0.821386
6,koi_impact,0.305554,0.694446,0.821386
7,koi_srho,0.671955,0.671955,0.821386
8,koi_ror,0.338378,0.661622,0.821386
9,koi_period,0.658234,0.658234,0.000000


In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.pipeline import make_pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    confusion_matrix
)

# Measurements
X = df_binary[valid_features].copy()

# Target:
# CONFIRMED = 1
# FALSE POSITIVE = 0
y = (
    df_binary["koi_disposition"] == "CONFIRMED"
).astype(int)

print("X shape:", X.shape)
print("y shape:", y.shape)
print("\nTarget distribution:")
print(y.value_counts())

X shape: (6696, 23)
y shape: (6696,)

Target distribution:
koi_disposition
0    3965
1    2731
Name: count, dtype: int64


In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

print("Training samples:", len(X_train))
print("Testing samples:", len(X_test))

Training samples: 5356
Testing samples: 1340


In [ ]:
model = make_pipeline(
    SimpleImputer(strategy="median"),
    StandardScaler(),
    LogisticRegression(
        max_iter=2000,
        random_state=42
    )
)

model.fit(X_train, y_train)

print("Model trained.")

Model trained.


In [ ]:
y_pred = model.predict(X_test)
y_prob = model.predict_proba(X_test)[:, 1]

print("First 10 predicted probabilities:")
print(y_prob[:10])

print("\nFirst 10 predictions:")
print(y_pred[:10])

First 10 predicted probabilities:
[0.85878016 0.92893194 0.90090102 0.88299455 0.00821401 0.89892085
 0.89480858 0.8515344  0.958201   0.90916099]

First 10 predictions:
[1 1 1 1 0 1 1 1 1 1]


In [ ]:
accuracy = accuracy_score(y_test, y_pred)
precision = precision_score(y_test, y_pred)
recall = recall_score(y_test, y_pred)
f1 = f1_score(y_test, y_pred)
auc = roc_auc_score(y_test, y_prob)

print(f"Accuracy : {accuracy:.3f}")
print(f"Precision: {precision:.3f}")
print(f"Recall   : {recall:.3f}")
print(f"F1 Score : {f1:.3f}")
print(f"ROC-AUC  : {auc:.3f}")

print("\nConfusion Matrix:")
print(confusion_matrix(y_test, y_pred))

Accuracy : 0.879
Precision: 0.827
Recall   : 0.890
F1 Score : 0.857
ROC-AUC  : 0.944

Confusion Matrix:
[[691 102]
 [ 60 487]]


In [ ]:
import pandas as pd

# Get the fitted Logistic Regression model
logreg = model.named_steps["logisticregression"]

# Get the feature names
feature_names = X_train.columns

# Get the learned coefficients
coefficients = logreg.coef_[0]

# Create a table
coef_df = pd.DataFrame({
    "feature": feature_names,
    "coefficient": coefficients,
    "absolute_coefficient": abs(coefficients)
})

# Sort by absolute coefficient
coef_df = (
    coef_df
    .sort_values("absolute_coefficient", ascending=False)
    .reset_index(drop=True)
)

display(coef_df)

,feature,coefficient,absolute_coefficient
0,koi_ror,-4.904721,4.904721
1,koi_prad,-3.744996,3.744996
2,koi_max_sngle_ev,-3.180275,3.180275
3,koi_depth,-2.441031,2.441031
4,koi_srad,-2.299597,2.299597
5,koi_teq,-1.771336,1.771336
6,koi_duration,-1.643803,1.643803
7,koi_max_mult_ev,-1.344482,1.344482
8,koi_model_snr,1.276858,1.276858
9,koi_smass,-1.042330,1.042330


In [ ]:
intercept = logreg.intercept_[0]

equation = f"z = {intercept:.4f}"

for feature, coefficient in zip(feature_names, coefficients):
    sign = "+" if coefficient >= 0 else "-"
    equation += f" {sign} {abs(coefficient):.4f}·{feature}"

print(equation)

z = -4.2500 - 0.4816·koi_period - 0.7343·koi_impact - 1.6438·koi_duration - 2.4410·koi_depth - 4.9047·koi_ror - 0.4075·koi_srho + 0.3720·koi_incl - 0.9794·koi_dor - 3.1803·koi_max_sngle_ev - 1.3445·koi_max_mult_ev + 1.2769·koi_model_snr - 0.8319·koi_num_transits + 0.2887·koi_bin_oedp_sig + 0.2736·koi_steff - 1.0254·koi_slogg + 0.9397·koi_smet - 2.2996·koi_srad - 1.0423·koi_smass - 0.1923·koi_kepmag - 3.7450·koi_prad + 0.6778·koi_sma - 1.7713·koi_teq + 0.2529·koi_insol
